# UIT DSC 2026 — Task 2 LegalQA độc lập

Notebook này chỉ dùng dữ liệu Task 2. Không dùng `index`, embedding, bi-encoder hay reranker đã train bằng Task 1. Lượt đầu dựng baseline BM25 hợp lệ; phần cuối có tùy chọn tạo dense embedding zero-shot từ model pretrained bên ngoài.

> Trước khi chạy từ GitHub, commit và push cả `Retrieval-LegalIR/qa_predict.py` lẫn notebook này. Cell clone có guard để dừng sớm nếu branch vẫn chứa pipeline QA cũ.


## 0. Chuẩn bị Google Drive

Đặt đúng ba file BTC vào `MyDrive/DSC2026/task2_qa/input/`:

```text
train.json
public-official.json
selected-contexts.zip
```

Artifact tạo ra sẽ được cache trong `MyDrive/DSC2026/task2_qa/artifacts/`, nên reconnect Colab không phải chunk/index lại.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/caubenq9999/LawRetrieval.git'
BRANCH = 'feat/qa_baseline'
ROOT = Path('/content/LawRetrieval')
DRIVE_ROOT = Path('/content/drive/MyDrive/DSC2026/task2_qa')
INPUT = DRIVE_ROOT / 'input'
CACHE = DRIVE_ROOT / 'artifacts'
QA_DIR = Path('/content/task2_qa')
WORK = Path('/content/task2_work')
EVAL_N = 200

CACHE.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
print('Input:', INPUT)
print('Cache:', CACHE)


## 1. Clone code và cài dependency


In [ ]:
import subprocess
import sys

if not (ROOT / '.git').exists():
    subprocess.run([
        'git', 'clone', '--branch', BRANCH, '--single-branch',
        REPO_URL, str(ROOT)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(ROOT / 'requirements.txt')
], check=True)
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print('Code revision:')
subprocess.run(['git', '-C', str(ROOT), 'log', '-1', '--oneline'], check=True)
help_text = subprocess.run([
    sys.executable, str(ROOT / 'Retrieval-LegalIR/qa_predict.py'), '--help'
], check=True, capture_output=True, text=True).stdout
assert '--retriever' in help_text, (
    'GitHub branch chưa có qa_predict.py độc lập cho Task 2. ' 
    'Hãy commit/push qa_predict.py và notebook mới rồi chạy lại cell này.'
)


## 2. Giải nén đúng dữ liệu Task 2


In [ ]:
import json
import shutil

required = [
    INPUT / 'train.json',
    INPUT / 'public-official.json',
    INPUT / 'selected-contexts.zip',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, 'Thiếu file trên Drive:\n' + '\n'.join(missing)

QA_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(INPUT / 'train.json', QA_DIR / 'train.json')
shutil.copy2(INPUT / 'public-official.json', QA_DIR / 'public-official.json')

context_dir = QA_DIR / 'selected-contexts'
if len(list(context_dir.glob('context_*.json'))) != 8532:
    subprocess.run([
        'unzip', '-q', '-o', str(INPUT / 'selected-contexts.zip'),
        '-d', str(QA_DIR)
    ], check=True)

train = json.loads((QA_DIR / 'train.json').read_text(encoding='utf-8-sig'))
public = json.loads((QA_DIR / 'public-official.json').read_text(encoding='utf-8-sig'))
contexts = list(context_dir.glob('context_*.json'))
assert len(train) == 7000, len(train)
assert len(public) == 1000, len(public)
assert len(contexts) == 8532, len(contexts)
print(f'Task 2 OK: {len(train)} train | {len(public)} public | {len(contexts)} contexts')


## 3. Tạo và cache `chunks_qa` + `index_qa`

Hai artifact này chỉ được dựng từ `Task 2/selected-contexts`. Lần chạy sau notebook lấy cache từ Drive.


In [ ]:
chunks = WORK / 'chunks_qa.jsonl'
cached_chunks = CACHE / 'chunks_qa.jsonl'
index_dir = WORK / 'index_qa'
cached_index = CACHE / 'index_qa.zip'

if not chunks.exists():
    if cached_chunks.exists():
        print('Copy chunks_qa từ Drive cache ...')
        shutil.copy2(cached_chunks, chunks)
    else:
        print('Chunk 8.532 context Task 2 ...')
        subprocess.run([
            sys.executable, str(ROOT / 'Chunking-LegalIR/chunker.py'),
            '--contexts', str(context_dir), '--out', str(chunks)
        ], check=True)
        shutil.copy2(chunks, cached_chunks)

if not (index_dir / 'meta.json').exists():
    if cached_index.exists():
        print('Giải nén index_qa từ Drive cache ...')
        subprocess.run([
            'unzip', '-q', '-o', str(cached_index), '-d', str(WORK)
        ], check=True)
    else:
        print('Build BM25 index Task 2 ...')
        subprocess.run([
            sys.executable, str(ROOT / 'Retrieval-LegalIR/bm25.py'), 'build',
            '--chunks', str(chunks), '--index', str(index_dir)
        ], check=True)
        subprocess.run([
            'zip', '-qr', str(cached_index), index_dir.name
        ], cwd=str(WORK), check=True)

meta = json.loads((index_dir / 'meta.json').read_text(encoding='utf-8'))
assert Path(meta['chunks_path']).resolve() == chunks.resolve()
print('chunks:', f'{chunks.stat().st_size / 1e6:.0f} MB')
print('index:', index_dir)
print('n_chunks:', meta['n_chunks'])


## 4. Đánh giá baseline BM25 độc lập

Lượt này không load embedding và không load reranker. `--sweep` so sánh 1–4 chunk và hai kiểu dựng đáp án trên cùng một lượt retrieval.


In [ ]:
qa_predict = ROOT / 'Retrieval-LegalIR/qa_predict.py'
subprocess.run([
    sys.executable, str(qa_predict),
    '--qa-dir', str(QA_DIR),
    '--index', str(index_dir),
    '--retriever', 'bm25',
    '--eval', '-n', str(EVAL_N), '--sweep'
], cwd=str(ROOT / 'Retrieval-LegalIR'), check=True)


## 5. Sinh submission BM25

Chỉ chạy sau khi đã chọn `--nchunk` và `--style` từ bảng validation phía trên. Giá trị mặc định dưới đây là 3 chunk + cite; sửa hai biến nếu sweep cho kết quả khác.


In [ ]:
NCHUNK = 3
STYLE = 'cite'
submission_prefix = WORK / 'submission_qa_bm25'

subprocess.run([
    sys.executable, str(qa_predict),
    '--qa-dir', str(QA_DIR),
    '--questions', str(QA_DIR / 'public-official.json'),
    '--index', str(index_dir),
    '--retriever', 'bm25',
    '--nchunk', str(NCHUNK), '--style', STYLE,
    '--out', str(submission_prefix)
], cwd=str(ROOT / 'Retrieval-LegalIR'), check=True)

submission_zip = Path(str(submission_prefix) + '.zip')
assert submission_zip.exists()
shutil.copy2(submission_zip, CACHE / submission_zip.name)
print('Submission:', submission_zip)


In [ ]:
from google.colab import files
files.download(str(submission_zip))


## 6. Tùy chọn: dense zero-shot riêng cho Task 2

Chỉ dùng model pretrained bên ngoài nếu thể lệ cho phép. Không dùng `v2-ft`, `v2-full-ft`, `emb_v2ft` hay bất kỳ checkpoint/artifact nào đã học từ Task 1. Cell encode có thể mất hơn một giờ trên T4 và tạo khoảng 1 GB vector.


In [ ]:
BUILD_DENSE = False  # đổi thành True khi muốn chạy
DENSE_MODEL = 'AITeamVN/Vietnamese_Embedding_V2'
emb_dir = WORK / 'emb_qa_v2'
cached_emb = CACHE / 'emb_qa_v2.zip'

if BUILD_DENSE:
    if not (emb_dir / 'meta.json').exists():
        if cached_emb.exists():
            subprocess.run([
                'unzip', '-q', '-o', str(cached_emb), '-d', str(WORK)
            ], check=True)
        else:
            subprocess.run([
                sys.executable, str(ROOT / 'Retrieval-LegalIR/encode.py'),
                '--chunks', str(chunks), '--out', str(emb_dir),
                '--model', DENSE_MODEL
            ], check=True)
            subprocess.run([
                'zip', '-qr', str(cached_emb), emb_dir.name
            ], cwd=str(WORK), check=True)
    print('Dense embedding ready:', emb_dir)
else:
    print('Bỏ qua dense. Đổi BUILD_DENSE=True để encode.')


In [ ]:
if BUILD_DENSE:
    subprocess.run([
        sys.executable, str(qa_predict),
        '--qa-dir', str(QA_DIR),
        '--index', str(index_dir),
        '--emb', str(emb_dir),
        '--retriever', 'hybrid', '--alpha', '0.5',
        '--eval', '-n', str(EVAL_N), '--sweep'
    ], cwd=str(ROOT / 'Retrieval-LegalIR'), check=True)
